[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/practice/advanced/01_advanced_ml_techniques.ipynb)

# Luyện tập nâng cao 1 — Kỹ thuật ML nâng cao, không rò rỉ

Các kỹ thuật mạnh nhưng **dễ rò rỉ** nhất: target encoding, stacking, đặc trưng tổng hợp theo thời gian; cùng các công cụ đánh giá nghiêm ngặt: permutation importance tự viết, khoảng tin cậy bootstrap theo cặp.

Chương liên quan: 15, 17, 24, 36, 39.

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/practice/advanced

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true

rng = np.random.default_rng(0)

**Bài 1 — Target encoding out-of-fold.** Cột `merchant_id` có 2.000 giá trị, **không liên quan** đến nhãn. (a) Mã hoá "ngây thơ" = trung bình nhãn **thô** của mỗi merchant tính trên toàn train, dùng cho cả train lẫn test (merchant chưa thấy → trung bình chung); (b) mã hoá **out-of-fold** (5-fold, `KFold(shuffle=True, random_state=0)`): mỗi dòng dùng trung bình nhãn tính từ các fold khác, làm trơn $\frac{n\bar y + m\mu}{n + m}$ với $m = 10$, $\mu$ = trung bình nhãn của phần dùng để tính. Viết `oof_target_encode(keys, y, m=10)`. Với (b), test mã hoá bằng trung bình **làm trơn** tính trên toàn train. Huấn luyện logistic trên `[signal, mã hoá]` cho mỗi cách, đo AUC test → `auc_naive`, `auc_oof`.

In [ ]:
n = 6000
merchant = rng.integers(0, 2000, n)
signal = rng.normal(size=n)
y1 = (rng.random(n) < 1 / (1 + np.exp(-signal))).astype(int)
tr, te = np.arange(4500), np.arange(4500, n)


def smoothed_means(keys, y, m=10):
    stats = pd.DataFrame({"k": keys, "y": y}).groupby("k").y.agg(["sum", "count"])
    mu = y.mean()
    return (stats["sum"] + m * mu) / (stats["count"] + m), mu


def oof_target_encode(keys, y, m=10):
    ### START CODE HERE ### (≈ 6 dòng)
    # STARTER: return np.zeros(len(keys))
    out = np.zeros(len(keys))
    for fit_idx, enc_idx in KFold(5, shuffle=True, random_state=0).split(keys):
        means, mu = smoothed_means(keys[fit_idx], y[fit_idx], m)
        out[enc_idx] = pd.Series(keys[enc_idx]).map(means).fillna(mu).values
    return out
    ### END CODE HERE ###


### START CODE HERE ### (≈ 9 dòng)
# STARTER: auc_naive, auc_oof = None, None
raw_means = pd.DataFrame({"k": merchant[tr], "y": y1[tr]}).groupby("k").y.mean()
naive_tr = pd.Series(merchant[tr]).map(raw_means).values
naive_te = pd.Series(merchant[te]).map(raw_means).fillna(y1[tr].mean()).values
auc_naive = roc_auc_score(y1[te], LogisticRegression().fit(np.column_stack([signal[tr], naive_tr]), y1[tr])
                          .predict_proba(np.column_stack([signal[te], naive_te]))[:, 1])
train_means, train_mu = smoothed_means(merchant[tr], y1[tr])
test_enc = pd.Series(merchant[te]).map(train_means).fillna(train_mu).values
auc_oof = roc_auc_score(y1[te], LogisticRegression().fit(np.column_stack([signal[tr], oof_target_encode(merchant[tr], y1[tr])]), y1[tr])
                        .predict_proba(np.column_stack([signal[te], test_enc]))[:, 1])
### END CODE HERE ###
auc_signal_only = roc_auc_score(y1[te], LogisticRegression().fit(signal[tr, None], y1[tr]).predict_proba(signal[te, None])[:, 1])
print(f"AUC test — mã hoá ngây thơ: {auc_naive:.3f} | out-of-fold: {auc_oof:.3f} | chỉ tín hiệu thật: {auc_signal_only:.3f}")

In [ ]:
check_less(abs(auc_oof - auc_signal_only), 0.01, "OOF: cột vô dụng không làm hại")
check_less(auc_naive, auc_oof - 0.1, "mã hoá ngây thơ: mô hình tin cột rò rỉ → kém hẳn trên test")

Với merchant chỉ có ~2 dòng, trung bình nhãn của chính nó gần như **là nhãn** → trên train, cột này "dự đoán" gần hoàn hảo, mô hình dồn trọng số vào nó; trên test, cột (trung bình nhãn của *những dòng khác*) chỉ là nhiễu nhưng vẫn được tin → hiệu năng sụp. OOF + làm trơn khiến cột phản ánh đúng mức thông tin thật (ở đây: không có).

**Bài 2 — Stacking đúng cách.** Hai mô hình nền: cây quyết định **không giới hạn độ sâu** (overfit) và kNN (k = 25). Meta-learner logistic học trên (a) dự đoán **in-sample** của mô hình nền trên train; (b) dự đoán **out-of-fold** (`cross_val_predict`, 5-fold). So AUC và log-loss trên test → `auc_stack_insample`, `auc_stack_oof`, `logloss_insample`, `logloss_oof`; và hệ số meta-learner gán cho cây trong mỗi cách (`tree_weight_insample`, `tree_weight_oof`).

In [ ]:
X2 = rng.normal(size=(4000, 6))
y2 = ((X2[:, 0] + X2[:, 1] ** 2 - 1 + rng.normal(0, 1, 4000)) > 0).astype(int)
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.3, random_state=0)
bases = [DecisionTreeClassifier(random_state=0), KNeighborsClassifier(25)]
### START CODE HERE ### (≈ 10 dòng)
in_sample = np.column_stack([b.fit(X2_tr, y2_tr).predict_proba(X2_tr)[:, 1] for b in bases])
oof = np.column_stack([cross_val_predict(b, X2_tr, y2_tr, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1] for b in bases])
test_level1 = np.column_stack([b.fit(X2_tr, y2_tr).predict_proba(X2_te)[:, 1] for b in bases])
meta_in = LogisticRegression().fit(in_sample, y2_tr)
meta_oof = LogisticRegression().fit(oof, y2_tr)
auc_stack_insample = roc_auc_score(y2_te, meta_in.predict_proba(test_level1)[:, 1])
auc_stack_oof = roc_auc_score(y2_te, meta_oof.predict_proba(test_level1)[:, 1])
tree_weight_insample, tree_weight_oof = meta_in.coef_[0][0], meta_oof.coef_[0][0]
logloss_insample = log_loss(y2_te, meta_in.predict_proba(test_level1)[:, 1])
logloss_oof = log_loss(y2_te, meta_oof.predict_proba(test_level1)[:, 1])
### END CODE HERE ###
print(f"meta in-sample: AUC {auc_stack_insample:.3f}, log-loss {logloss_insample:.3f}, trọng số cây {tree_weight_insample:.2f}")
print(f"meta OOF      : AUC {auc_stack_oof:.3f}, log-loss {logloss_oof:.3f}, trọng số cây {tree_weight_oof:.2f}")

In [ ]:
check_greater(tree_weight_insample, 3 * tree_weight_oof, "in-sample: meta-learner tin nhầm cây overfit")
check_greater(auc_stack_oof, auc_stack_insample + 0.005, "OOF stacking xếp hạng tốt hơn trên test")
check_less(logloss_oof, 0.8 * logloss_insample, "in-sample stacking cho xác suất quá tự tin (log-loss tệ hơn hẳn)")

Trên train, cây không giới hạn độ sâu dự đoán **hoàn hảo** (xác suất 0/1 khớp nhãn) → meta-learner học "tin cây tuyệt đối". Trên test, cây overfit sai nhiều mà vẫn được tin → xác suất cực đoan và sai, log-loss tăng vọt; thứ hạng (AUC) cũng kém hơn. Dự đoán out-of-fold phản ánh đúng chất lượng **ngoài mẫu** của từng mô hình nền — điều kiện bắt buộc của stacking (chương 24).

**Bài 3 — Đặc trưng tổng hợp đúng thời điểm.** Bảng giao dịch `tx` (khách, thời điểm, số tiền) và bảng nhãn `labels` (khách, thời điểm dự đoán). Với mỗi dòng nhãn, tính **tổng số tiền giao dịch trong 7 ngày trước thời điểm dự đoán** (không gồm giao dịch tại hoặc sau thời điểm đó) → cột `spend_7d`. Viết `spend_last_7_days(labels, tx)`.

In [ ]:
tx = pd.DataFrame({"customer": ["a", "a", "a", "b", "b", "a"],
                   "time": pd.to_datetime(["2024-01-01", "2024-01-05", "2024-01-09", "2024-01-03", "2024-01-10", "2024-01-10"]),
                   "amount": [100, 50, 30, 200, 70, 999]})
labels = pd.DataFrame({"customer": ["a", "b", "a"], "time": pd.to_datetime(["2024-01-10", "2024-01-10", "2024-01-06"])})


def spend_last_7_days(labels, tx):
    ### START CODE HERE ### (≈ 6 dòng)
    # STARTER: return labels.assign(spend_7d=0.0)
    merged = labels.reset_index().merge(tx, on="customer", suffixes=("", "_tx"))
    window = (merged.time_tx < merged.time) & (merged.time_tx >= merged.time - pd.Timedelta(days=7))
    spend = merged[window].groupby("index").amount.sum()
    out = labels.copy()
    out["spend_7d"] = spend.reindex(range(len(labels)), fill_value=0).astype(float).values
    return out
    ### END CODE HERE ###


print(spend_last_7_days(labels, tx))

In [ ]:
check_equal(list(spend_last_7_days(labels, tx).spend_7d), [80.0, 200.0, 150.0], "tổng 7 ngày trước, không gồm thời điểm dự đoán và tương lai")
check_true(999 not in spend_last_7_days(labels, tx).spend_7d.values, "giao dịch cùng thời điểm dự đoán bị loại")

**Bài 4 — Permutation importance tự viết.** `perm_importance(model, X, y, n_repeats=5, seed=0)`: với mỗi cột, xáo trộn cột đó (`np.random.default_rng(seed)`, mỗi lần lặp một hoán vị mới), đo **mức giảm ROC-AUC**; trả về mảng trung bình theo cột. So thứ hạng với `sklearn.inspection.permutation_importance`.

In [ ]:
X4 = rng.normal(size=(3000, 5))
y4 = (rng.random(3000) < 1 / (1 + np.exp(-(2 * X4[:, 0] - 1 * X4[:, 1] + 0.3 * X4[:, 2])))).astype(int)
X4_tr, X4_te, y4_tr, y4_te = train_test_split(X4, y4, test_size=0.4, random_state=0)
forest = RandomForestClassifier(200, min_samples_leaf=5, random_state=0, n_jobs=-1).fit(X4_tr, y4_tr)


def perm_importance(model, X, y, n_repeats=5, seed=0):
    ### START CODE HERE ### (≈ 9 dòng)
    # STARTER: return np.zeros(X.shape[1])
    r = np.random.default_rng(seed)
    base = roc_auc_score(y, model.predict_proba(X)[:, 1])
    drops = np.zeros(X.shape[1])
    for j in range(X.shape[1]):
        for _ in range(n_repeats):
            Xp = X.copy()
            Xp[:, j] = r.permutation(Xp[:, j])
            drops[j] += base - roc_auc_score(y, model.predict_proba(Xp)[:, 1])
    return drops / n_repeats
    ### END CODE HERE ###


mine = perm_importance(forest, X4_te, y4_te)
ref = permutation_importance(forest, X4_te, y4_te, scoring="roc_auc", n_repeats=5, random_state=0).importances_mean
print(np.round(mine, 4), np.round(ref, 4))

In [ ]:
check_equal(list(np.argsort(-mine)[:3]), [0, 1, 2], "thứ hạng đúng: x0 > x1 > x2")
check_close(mine, ref, tol=0.01, message="gần scikit-learn")
check_less(np.abs(mine[3:]).max(), 0.01, "đặc trưng nhiễu ≈ 0")

**Bài 5 — Khoảng tin cậy bootstrap theo cặp.** Hai mô hình trên cùng tập test: `p_a` (logistic) và `p_b` (rừng). Viết `paired_bootstrap(y, p_a, p_b, n_boot=1000, seed=0)` → `(mean_diff, lo, hi)` cho chênh lệch AUC (A − B), mỗi lần lấy mẫu lại **cùng chỉ số** cho cả hai mô hình. Kết luận được A tốt hơn B không?

In [ ]:
p_a = LogisticRegression().fit(X4_tr, y4_tr).predict_proba(X4_te)[:, 1]
p_b = forest.predict_proba(X4_te)[:, 1]


def paired_bootstrap(y, p_a, p_b, n_boot=1000, seed=0):
    ### START CODE HERE ### (≈ 7 dòng)
    # STARTER: return 0.0, 0.0, 0.0
    r = np.random.default_rng(seed)
    y = np.asarray(y)
    diffs = []
    for _ in range(n_boot):
        i = r.integers(0, len(y), len(y))
        diffs.append(roc_auc_score(y[i], p_a[i]) - roc_auc_score(y[i], p_b[i]))
    return float(np.mean(diffs)), float(np.quantile(diffs, 0.025)), float(np.quantile(diffs, 0.975))
    ### END CODE HERE ###


mean_diff, lo, hi = paired_bootstrap(y4_te, p_a, p_b)
print(f"AUC A − B = {mean_diff:+.4f}, KTC 95% [{lo:+.4f}, {hi:+.4f}]")

In [ ]:
check_true(lo < mean_diff < hi, "KTC bao quanh trung bình")
check_greater(lo, 0, "dữ liệu sinh từ mô hình logistic → logistic tốt hơn rừng một cách có ý nghĩa")
check_less(hi - lo, 0.03, "so sánh theo cặp cho KTC hẹp")